In [1]:
import os

import requests
from dotenv import load_dotenv
from openai import AsyncOpenAI
from pydantic import BaseModel, Field

from agents import (
    Agent,
    GuardrailFunctionOutput,
    OpenAIChatCompletionsModel,
    Runner,
    function_tool,
    output_guardrail,
    trace,
)

load_dotenv(override=True)

True

In [2]:
key_names = ["OPENAI_API_KEY", "GOOGLE_API_KEY", "GROQ_API_KEY"]

for key_name in key_names:
    print(f"{key_name} set:", bool(os.getenv(key_name)))

OPENAI_API_KEY set: True
GOOGLE_API_KEY set: True
GROQ_API_KEY set: True


In [3]:
instructions = """
You are a sales agent working for CloudWatchly,
a company that provides a SaaS tool for monitoring cloud costs and flagging waste, powered by AI.
You write compelling sales emails that are likely to get a response.
"""

In [4]:
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
GROQ_BASE_URL = "https://api.groq.com/openai/v1"

In [5]:
google_api_key = os.getenv("GOOGLE_API_KEY")
groq_api_key = os.getenv("GROQ_API_KEY")

gemini_client = AsyncOpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)
groq_client = AsyncOpenAI(base_url=GROQ_BASE_URL, api_key=groq_api_key)

In [6]:
gemini_model = OpenAIChatCompletionsModel(model="gemini-3.1-flash-lite", openai_client=gemini_client)
oss_model = OpenAIChatCompletionsModel(model="openai/gpt-oss-120b", openai_client=groq_client)

In [7]:
sales_agent_gemini = Agent(name="Gemini Sales Agent", instructions=instructions, model=gemini_model)
sales_agent_oss = Agent(name="GPT-OSS Sales Agent", instructions=instructions, model=oss_model)

In [8]:
result = await Runner.run(sales_agent_gemini, "Write a cold sales email")
print(result.final_output)

Subject: Is your cloud bill growing faster than your revenue?

Hi [Prospect Name],

Most engineering and finance teams I speak with at [Prospect Company]’s stage face the same "cloud trap": you migrate to the cloud to scale quickly, but you end up with a monthly bill that’s inflated by forgotten instances, over-provisioned resources, and idle dev environments.

At **CloudWatchly**, we’ve built an AI-driven platform that doesn’t just show you what you’re spending—it tells you exactly where you’re bleeding money and automates the cleanup.

Unlike standard dashboards that just alert you after the damage is done, we integrate directly into your workflow to:
*   **Flag hidden waste:** Identify zombie resources that legacy tools miss.
*   **Predict spikes:** Get ahead of cost anomalies before they hit your P&L.
*   **Automate savings:** Enable "one-click" rightsizing that your engineers will actually use.

We recently helped a company in [Prospect's Industry] reduce their monthly cloud spend

In [9]:
result = await Runner.run(sales_agent_oss, "Write a cold sales email")
print(result.final_output)

**Subject:** Cut cloud‑cost waste by up to 30 %—no extra engineering effort  

---  

Hi **[First Name]**,

I’ve noticed that many fast‑growing teams like **[Company]** spend up to 30 % of their cloud budget on idle resources, orphaned volumes, and over‑provisioned instances. The hidden waste is easy to miss, especially when you’re focused on delivering new features.

That’s why we built **CloudWatchly**—an AI‑powered SaaS platform that continuously monitors your entire cloud footprint, automatically flags inefficiencies, and provides one‑click remediation recommendations.

**What you’ll get:**
- **Real‑time cost visibility** across AWS, Azure, and GCP in a single dashboard.  
- **AI‑driven waste detection** that surfaces idle VMs, under‑utilized storage, and forgotten test environments before the bill hits.  
- **Actionable insights** with auto‑generated tickets or scripts to remediate – no manual digging required.  
- **ROI in weeks:** our customers see an average **22 % reduction** 

## a manager that uses agents running on different models


In [10]:
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"


def push(message):
    print(f"Push: {message[:100]}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message[:1000]}
    response = requests.post(pushover_url, data=payload, timeout=10)
    print("Status:", response.status_code)


def send_message(subject, body):
    push(f"Subject: {subject}\n\n{body}")

In [11]:
@function_tool
def send_email_tool(subject: str, body: str) -> str:
    """
    Send the chosen sales email to the user as a notification

    Args:
        subject: The subject line of the email
        body: The body of the email as plain text
    """
    send_message(subject, body)
    return "Email sent successfully"

In [12]:
sales_agent_openai = Agent(name="OpenAI Sales Agent", instructions=instructions, model="gpt-5.4-mini")

description = "Use this tool to write a sales email. In the input, just instruct it to write a sales email."

tool1 = sales_agent_openai.as_tool(tool_name="sales_agent_1", tool_description=description)
tool2 = sales_agent_gemini.as_tool(tool_name="sales_agent_2", tool_description=description)
tool3 = sales_agent_oss.as_tool(tool_name="sales_agent_3", tool_description=description)

manager_tools = [tool1, tool2, tool3, send_email_tool]

In [13]:
manager_instructions = """
You are a Sales Manager at CloudWatchly. Your goal is to find the single best cold sales email using the sales_agent tools.
"""

manager_task = """
Follow these steps:

1. Generate Drafts: Use each of the three sales_agent tools to generate different email drafts.
Just instruct each to write a sales email; no further details are needed.
Do not proceed until all three drafts are ready, one from each tool.

2. Evaluate and Select: Review the drafts and choose the single best email using your judgment of which one is most effective.

3. Use your tool to send the best email (and only the best email) to the user. Only send 1 email.
"""

sales_manager = Agent(
    name="Sales Manager",
    instructions=manager_instructions,
    tools=manager_tools,
    model="gpt-5.4-mini",
)

In [14]:
with trace("Sales Manager across different models"):
    result = await Runner.run(sales_manager, manager_task)

print(result.final_output)

Push: Subject: Cut cloud waste before it hits next month’s bill

Hi [First Name],

Cloud bills have a way 
Status: 200
Sent.


In [15]:
class EmailReview(BaseModel):
    is_professional: bool = Field(description="Whether the email is professional and appropriate")
    number_of_sentences: int = Field(description="The number of sentences in the body of the email, not including the greeting and signature")
    contains_placeholders: bool = Field(description="Whether the email contains placeholders such as [first_name]")

In [16]:
EmailReview.model_json_schema()

{'properties': {'is_professional': {'description': 'Whether the email is professional and appropriate',
   'title': 'Is Professional',
   'type': 'boolean'},
  'number_of_sentences': {'description': 'The number of sentences in the body of the email, not including the greeting and signature',
   'title': 'Number Of Sentences',
   'type': 'integer'},
  'contains_placeholders': {'description': 'Whether the email contains placeholders such as [first_name]',
   'title': 'Contains Placeholders',
   'type': 'boolean'}},
 'required': ['is_professional',
  'number_of_sentences',
  'contains_placeholders'],
 'title': 'EmailReview',
 'type': 'object'}

In [17]:
sample_email = """
Hi [first_name],

I'm hitting you up to see if you'd like to buy our product. It's really great. You'll miss out if you don't buy it.

Laters.

Siva
"""

In [18]:
checker = Agent(
    name="Checker",
    instructions="You review potential sales emails",
    model="gpt-5.4-mini",
    output_type=EmailReview,
)

result = await Runner.run(checker, sample_email)

In [19]:
review = result.final_output
review

EmailReview(is_professional=False, number_of_sentences=3, contains_placeholders=True)

In [20]:
review.is_professional

False

In [21]:
review.contains_placeholders

True

## guardrails with the SDK


In [22]:
@output_guardrail
async def email_guardrail(ctx, agent, message):
    result = await Runner.run(checker, message, context=ctx.context)
    review = result.final_output

    is_problem = review.contains_placeholders or not review.is_professional

    return GuardrailFunctionOutput(
        output_info={"review": review},
        tripwire_triggered=is_problem,
    )

In [23]:
cowboy_instructions = instructions + "\nSpeak like a cowboy"

sales_agent_cowboy = Agent(
    name="Cowboy",
    instructions=cowboy_instructions,
    model=gemini_model,
    output_guardrails=[email_guardrail],
)

In [24]:
result = await Runner.run(sales_agent_cowboy, "Write a cold sales email")
result.final_output

OutputGuardrailTripwireTriggered: Guardrail OutputGuardrail triggered tripwire

In [25]:
from agents import OutputGuardrailTripwireTriggered

try:
    result = await Runner.run(sales_agent_cowboy, "Write a cold sales email")
    print("Passed the guardrail:")
    print(result.final_output)
except OutputGuardrailTripwireTriggered:
    print("Guardrail tripped: the email was blocked")

Guardrail tripped: the email was blocked


## the same guardrail as plain code

In [26]:
simple_cowboy = Agent(
    name="Simple Cowboy",
    instructions=cowboy_instructions,
    model=gemini_model,
)

result = await Runner.run(simple_cowboy, "Write a cold sales email")
email = result.final_output
print(email)

Subject: Taming those runaway cloud costs, partner 🤠

Howdy,

I reckon I don’t need to tell you that cloud bills have a way of stampeding right over a budget when you aren't lookin’. It’s easy to lose track of where the gold is leaking out of the bucket when your infrastructure gets as big as a Texas prairie.

I’m reaching out from **CloudWatchly**. We’ve built a sharp-eyed AI sidekick that monitors your cloud environment 24/7. It’s got a nose for waste, sniffin’ out idle resources and over-provisioned services that are burnin’ through your cash faster than a wildfire in August.

Most folks we work with see their monthly spend drop by about 20-30% within the first month—all without liftin’ a finger or breakin’ a sweat.

I’d love to show you how we can help you round up those costs and put that money back where it belongs: in your pocket. 

You got a few minutes later this week to talk shop? I promise to keep it shorter than a quick-draw duel.

Best trails,

[Your Name]
CloudWatchly


In [27]:
check = await Runner.run(checker, email)
review = check.final_output

if not review.is_professional or review.contains_placeholders:
    print("The email is not professional or has placeholders, so it will not be sent")
else:
    print("Email is good")

The email is not professional or has placeholders, so it will not be sent


In [28]:
async def write_checked_email(agent, max_tries=2):
    for attempt in range(max_tries):
        result = await Runner.run(agent, "Write a cold sales email")
        email = result.final_output

        check = await Runner.run(checker, email)
        review = check.final_output

        if review.is_professional and not review.contains_placeholders:
            return email

        print(f"Attempt {attempt + 1} failed the check")

    return None

the exercise. Add an input guardrail and use structured outputs for email generation

In [29]:
class SalesEmail(BaseModel):
    subject: str = Field(description="A short, compelling subject line")
    body: str = Field(description="The body of the email as plain text, with a greeting and a signature")

In [30]:
structured_writer = Agent(
    name="Structured Writer",
    instructions=instructions,
    model="gpt-5.4-mini",
    output_type=SalesEmail,
)

result = await Runner.run(structured_writer, "Write a cold sales email")
email = result.final_output

print(email.subject)
print(email.body)

Quick idea to reduce cloud waste
Hi,

I’m reaching out from CloudWatchly. We help teams monitor cloud spend, flag waste automatically, and spot savings opportunities with AI before costs get out of hand.

If you’re already using AWS, Azure, or GCP, we may be able to surface unused resources, idle services, and other hidden waste without adding more manual work for your team.

Would you be open to a quick 15-minute chat next week to see if there’s a fit?

Best,
[Your Name]
CloudWatchly


In [31]:
check = await Runner.run(checker, email.body)
review = check.final_output

if review.is_professional and not review.contains_placeholders:
    send_message(email.subject, email.body)
    print("Sent")
else:
    print("Blocked by the check")

Blocked by the check


In [32]:
class InputReview(BaseModel):
    is_misleading_request: bool = Field(description="Whether the request asks for a deceptive, false or spammy email")
    reason: str = Field(description="A one-sentence reason for the decision")


input_checker = Agent(
    name="Input Checker",
    instructions="You review requests to write sales emails and flag any that ask for deception, false claims or spam.",
    model="gpt-5.4-mini",
    output_type=InputReview,
)